# Case Study 4: Social Media Sentiment Analysis

**Objective:** Analyze tweets for sentiment by cleaning noisy text data (emojis, hashtags, irrelevant characters), transforming unstructured content into a structured format, and handling large-scale streaming data.

---
## Questions Covered
1. How to perform **data cleaning** for text data?
2. **String manipulation** techniques used in preprocessing.
3. How to **transform unstructured data** into structured format?
4. Methods to handle **large volumes of streaming data**.


## Q1: Data Cleaning for Text Data

In [ ]:
import re
import string
import pandas as pd
import numpy as np

sample_tweets = [
    "I love this product! 😍 #amazing @brand https://t.co/example &amp; it's great!!!",
    "Worst service EVER 😡😡 #terrible #bad <b>HTML tags</b> here...",
    "  Just a normal tweet with extra   spaces  ",
    "RT @user: Retweet example with #hashtag and URL http://example.com",
    "Mixed CASE tweet with 1234 numbers and special chars @#$%",
    "café résumé naïve — unicode and em-dash — characters",
]

raw_df = pd.DataFrame({"raw_tweet": sample_tweets})
print("Raw Tweets:")
print(raw_df["raw_tweet"].to_string())


In [ ]:
def remove_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_html(text):
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'&[a-z]+;', '', text)
    return text

def remove_retweet_mentions(text):
    text = re.sub(r'^RT\s+', '', text)
    text = re.sub(r'@\w+', '', text)
    return text

def remove_emojis(text):
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002600-\U000027BF"
        "]+", flags=re.UNICODE
    )
    return emoji_pattern.sub('', text)

def remove_special_chars(text):
    return re.sub(r'[^a-zA-Z0-9\s#@]', '', text)

def normalize_whitespace(text):
    return re.sub(r'\s+', ' ', text).strip()

def full_clean_pipeline(text):
    text = remove_urls(text)
    text = remove_html(text)
    text = remove_retweet_mentions(text)
    text = remove_emojis(text)
    text = remove_special_chars(text)
    text = normalize_whitespace(text)
    return text

raw_df["cleaned_tweet"] = raw_df["raw_tweet"].apply(full_clean_pipeline)
print("Cleaned Tweets:")
print(raw_df[["raw_tweet", "cleaned_tweet"]].to_string())


## Q2: String Manipulation Techniques in Preprocessing

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

def lowercase(text):
    return text.lower()

def expand_contractions(text):
    contractions = {
        "can't": "cannot", "won't": "will not", "it's": "it is",
        "i'm": "i am", "i've": "i have", "you're": "you are",
        "they're": "they are", "that's": "that is", "isn't": "is not",
    }
    for contraction, expansion in contractions.items():
        text = text.replace(contraction, expansion)
    return text

def tokenize(text):
    return word_tokenize(text)

def remove_stopwords(tokens):
    stop_words = set(stopwords.words('english'))
    return [t for t in tokens if t not in stop_words]

def stem_tokens(tokens):
    stemmer = PorterStemmer()
    return [stemmer.stem(t) for t in tokens]

def lemmatize_tokens(tokens):
    lemmatizer = WordNetLemmatizer()
    return [lemmatizer.lemmatize(t) for t in tokens]

def remove_punctuation_tokens(tokens):
    return [t for t in tokens if t not in string.punctuation]

sample = "I'm loving this! The products are absolutely amazing and they're helping."

print("Original        :", sample)
text = lowercase(sample)
print("Lowercased      :", text)
text = expand_contractions(text)
print("Contractions    :", text)
tokens = tokenize(text)
print("Tokens          :", tokens)
tokens = remove_punctuation_tokens(tokens)
print("No Punctuation  :", tokens)
tokens_ns = remove_stopwords(tokens)
print("No Stopwords    :", tokens_ns)
stemmed = stem_tokens(tokens_ns)
print("Stemmed         :", stemmed)
lemmatized = lemmatize_tokens(tokens_ns)
print("Lemmatized      :", lemmatized)


In [ ]:
def extract_hashtags(text):
    return re.findall(r'#(\w+)', text)

def extract_mentions(text):
    return re.findall(r'@(\w+)', text)

def extract_emojis(text):
    emoji_pattern = re.compile(
        "[" u"\U0001F600-\U0001F64F" u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF" u"\U00002600-\U000027BF" "]+",
        flags=re.UNICODE
    )
    return emoji_pattern.findall(text)

tweet = "Great product! 😍❤️ #amazing #mustbuy @brand @user https://t.co/ex"
print("Hashtags :", extract_hashtags(tweet))
print("Mentions :", extract_mentions(tweet))
print("Emojis   :", extract_emojis(tweet))


## Q3: Transforming Unstructured Data into Structured Format

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from textblob import TextBlob

tweets_extended = [
    "I love this product! 😍 #amazing @brand https://t.co/ex &amp; great!!!",
    "Worst service EVER 😡 #terrible #bad HTML tags here...",
    "Just a normal tweet with extra spaces",
    "RT @user: Retweet example with #hashtag http://example.com",
    "Mixed CASE tweet with numbers and special chars",
    "Great experience today #happy #positive feeling wonderful",
    "Absolutely terrible, never buying again #disappointed",
]

df = pd.DataFrame({"raw_tweet": tweets_extended})
df["cleaned"]   = df["raw_tweet"].apply(full_clean_pipeline)
df["hashtags"]  = df["raw_tweet"].apply(extract_hashtags)
df["mentions"]  = df["raw_tweet"].apply(extract_mentions)
df["emojis"]    = df["raw_tweet"].apply(extract_emojis)
df["char_count"]  = df["cleaned"].apply(len)
df["word_count"]  = df["cleaned"].apply(lambda x: len(x.split()))
df["has_url"]     = df["raw_tweet"].apply(lambda x: bool(re.search(r'https?://', x)))
df["is_retweet"]  = df["raw_tweet"].apply(lambda x: x.startswith("RT"))
df["hashtag_count"] = df["hashtags"].apply(len)
df["mention_count"] = df["mentions"].apply(len)
df["emoji_count"]   = df["emojis"].apply(len)

print("Structured Features:")
print(df[["cleaned", "char_count", "word_count", "hashtag_count", "mention_count",
          "emoji_count", "has_url", "is_retweet"]].to_string())


In [ ]:
def get_sentiment_features(text):
    blob = TextBlob(text)
    polarity    = blob.sentiment.polarity
    subjectivity = blob.sentiment.subjectivity
    if polarity > 0.1:
        label = "Positive"
    elif polarity < -0.1:
        label = "Negative"
    else:
        label = "Neutral"
    return pd.Series({
        "polarity": round(polarity, 4),
        "subjectivity": round(subjectivity, 4),
        "sentiment_label": label
    })

sentiment_df = df["cleaned"].apply(get_sentiment_features)
df = pd.concat([df, sentiment_df], axis=1)
print("Sentiment Features:")
print(df[["cleaned", "polarity", "subjectivity", "sentiment_label"]].to_string())


In [ ]:
tfidf = TfidfVectorizer(max_features=10, ngram_range=(1, 2))
tfidf_matrix = tfidf.fit_transform(df["cleaned"])
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())
print("TF-IDF Feature Matrix (top 10 features):")
print(tfidf_df.to_string())


In [ ]:
cv = CountVectorizer(max_features=10)
bow_matrix = cv.fit_transform(df["cleaned"])
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=cv.get_feature_names_out())
print("Bag-of-Words Feature Matrix:")
print(bow_df.to_string())


In [ ]:
final_structured = pd.concat([
    df[["cleaned", "char_count", "word_count", "hashtag_count",
        "mention_count", "emoji_count", "has_url", "is_retweet",
        "polarity", "subjectivity", "sentiment_label"]],
    tfidf_df
], axis=1)

print("Final Structured DataFrame Shape:", final_structured.shape)
print("\nColumn Names:")
print(list(final_structured.columns))
print("\nSample Rows:")
print(final_structured.head(3).to_string())


## Q4: Handling Large Volumes of Streaming Data

In [ ]:
import time
import queue
import threading
import random
from collections import defaultdict, deque
from datetime import datetime

SAMPLE_TWEETS = [
    "Loving this new feature! #awesome #tech",
    "Terrible experience today #bad @support",
    "Just had coffee ☕ #morning",
    "Amazing product launch! #excited 😊",
    "Service down again #disappointed",
    "Great customer support @brand #happy",
    "The worst update ever #angry",
    "Beautiful day outside #nature #happy",
]

def simulate_tweet_stream(tweet_queue, n=20, delay=0.05):
    for i in range(n):
        tweet = {
            "id":        i,
            "text":      random.choice(SAMPLE_TWEETS),
            "timestamp": datetime.now().isoformat(),
            "user":      f"user_{random.randint(1, 100)}"
        }
        tweet_queue.put(tweet)
        time.sleep(delay)
    tweet_queue.put(None)

print("Tweet stream simulator defined.")


In [ ]:
class SentimentAggregator:
    def __init__(self, window_size=10):
        self.window_size   = window_size
        self.recent_tweets = deque(maxlen=window_size)
        self.total_counts  = defaultdict(int)
        self.processed     = 0

    def process(self, tweet):
        text      = full_clean_pipeline(tweet["text"])
        blob      = TextBlob(text)
        polarity  = blob.sentiment.polarity
        label     = "Positive" if polarity > 0.1 else ("Negative" if polarity < -0.1 else "Neutral")

        record = {**tweet, "cleaned": text, "polarity": round(polarity, 4), "sentiment": label}
        self.recent_tweets.append(record)
        self.total_counts[label] += 1
        self.processed += 1
        return record

    def window_stats(self):
        if not self.recent_tweets:
            return {}
        labels = [t["sentiment"] for t in self.recent_tweets]
        polarities = [t["polarity"] for t in self.recent_tweets]
        return {
            "window_size":   len(self.recent_tweets),
            "avg_polarity":  round(np.mean(polarities), 4),
            "positive_pct":  round(labels.count("Positive") / len(labels) * 100, 1),
            "negative_pct":  round(labels.count("Negative") / len(labels) * 100, 1),
            "neutral_pct":   round(labels.count("Neutral")  / len(labels) * 100, 1),
        }

aggregator = SentimentAggregator(window_size=5)
print("SentimentAggregator ready.")


In [ ]:
tweet_queue   = queue.Queue()
results       = []

producer = threading.Thread(target=simulate_tweet_stream, args=(tweet_queue, 20, 0.02))
producer.start()

print("Processing stream...\n")
while True:
    tweet = tweet_queue.get()
    if tweet is None:
        break
    record = aggregator.process(tweet)
    results.append(record)
    if aggregator.processed % 5 == 0:
        stats = aggregator.window_stats()
        print(f"[Tweet {aggregator.processed:>3}] Sentiment: {record['sentiment']:<9} | "
              f"Polarity: {record['polarity']:>6} | "
              f"Window → Pos: {stats['positive_pct']}% | "
              f"Neg: {stats['negative_pct']}% | "
              f"Neu: {stats['neutral_pct']}%")

producer.join()
print(f"\nTotal tweets processed: {aggregator.processed}")
print("Overall distribution:", dict(aggregator.total_counts))


In [ ]:
def chunk_processor(tweet_list, chunk_size=5):
    results = []
    for i in range(0, len(tweet_list), chunk_size):
        chunk = tweet_list[i:i + chunk_size]
        chunk_df = pd.DataFrame(chunk)
        chunk_df["cleaned"]   = chunk_df["text"].apply(full_clean_pipeline)
        chunk_df["polarity"]  = chunk_df["cleaned"].apply(lambda x: TextBlob(x).sentiment.polarity)
        chunk_df["sentiment"] = chunk_df["polarity"].apply(
            lambda p: "Positive" if p > 0.1 else ("Negative" if p < -0.1 else "Neutral")
        )
        results.append(chunk_df)
        print(f"Chunk {i//chunk_size + 1}: processed {len(chunk)} tweets | "
              f"Avg polarity: {chunk_df['polarity'].mean():.4f}")
    return pd.concat(results, ignore_index=True)

print("=== Chunk-based Batch Processing ===")
stream_df = chunk_processor(results, chunk_size=5)
print(f"\nTotal processed: {len(stream_df)} tweets")


In [ ]:
print("=== Final Summary ===\n")
print("Sentiment Distribution:")
print(stream_df["sentiment"].value_counts().to_string())
print(f"\nAverage Polarity : {stream_df['polarity'].mean():.4f}")
print(f"Median Polarity  : {stream_df['polarity'].median():.4f}")
print(f"Max Polarity     : {stream_df['polarity'].max():.4f}")
print(f"Min Polarity     : {stream_df['polarity'].min():.4f}")
print("\nSample Processed Tweets:")
print(stream_df[["text", "cleaned", "polarity", "sentiment"]].head(5).to_string())


---
## Summary

| Question | Key Techniques |
|---|---|
| **Data Cleaning** | URL removal, HTML stripping, emoji removal, special char removal, whitespace normalization |
| **String Manipulation** | Lowercasing, contraction expansion, tokenization, stopword removal, stemming, lemmatization |
| **Structured Transformation** | Feature engineering (char/word count, flags), TF-IDF, Bag-of-Words, sentiment scores |
| **Streaming Data** | Producer-consumer queues, sliding window aggregation, chunk-based batch processing, multithreading |
